In [1]:
import torch
import pyro
import pyro.distributions as dist

num_samples = 100_000

### Multiplying individual frequencies

In [2]:
human_match_count = 29
human_reference_size = 1148

dog_match_count = 2
dog_reference_size = 78

human_frequency = human_match_count / human_reference_size
dog_frequency = dog_match_count / dog_reference_size

print(f"Human frequency: {human_frequency}")
print(f"Dog frequency: {dog_frequency}")

multiplied_frequencies = human_frequency * dog_frequency
print(f"Multiplied frequencies: {multiplied_frequencies:3e}")

Human frequency: 0.025261324041811847
Dog frequency: 0.02564102564102564
Multiplied frequencies: 6.477263e-04


### Multiplying samples from appropriate Beta distributions

In [3]:
human_alpha = 1 + human_match_count
human_beta = 1 + (human_reference_size - human_match_count)

dog_alpha = 1 + dog_match_count
dog_beta = 1 + (dog_reference_size - dog_match_count)

human_dist = dist.Beta(human_alpha, human_beta)
dog_dist = dist.Beta(dog_alpha, dog_beta)

torch.manual_seed(5)
human_samples = human_dist.sample((num_samples,))
dog_samples = dog_dist.sample((num_samples,))

# pointwise product, MC estimate
multiplied_samples = human_samples * dog_samples

print("Posterior human mean:", human_samples.mean().item())
print("Posterior dog mean:", dog_samples.mean().item())
print(f"Posterior product mean, MC estimate: {multiplied_samples.mean().item():.3e}")

Posterior human mean: 0.026077060028910637
Posterior dog mean: 0.037529412657022476
Posterior product mean, MC estimate: 9.784e-04


Because $P$ and $Q$ are independent,  
$$
\mathbb{E}[Z] = \mathbb{E}[P] \, \mathbb{E}[Q].
$$

For a $\mathrm{Beta}(\alpha,\beta)$ random variable,  
$$
\mathbb{E}[P] = \frac{\alpha}{\alpha + \beta}.
$$

Therefore,  
$$
\mathbb{E}[Z] = \frac{\alpha_1}{\alpha_1 + \beta_1} \cdot \frac{\alpha_2}{\alpha_2 + \beta_2}.
$$


In [4]:
joint_analytic = (human_alpha / (human_alpha + human_beta)) * (dog_alpha / (dog_alpha + dog_beta))
print(f"Analytic product mean: {joint_analytic:.3e}")

Analytic product mean: 9.783e-04
